In [0]:

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import current_timestamp, input_file_name
import logging

#input_path="/Volumes/workspace/default/vol1/raw_sales_transactions.csv"
#output_path="/workspace/bronze/bronze_sales_transactions.delta" 


logger = logging.getLogger(__name__)


def load_raw_transactions(
    spark: SparkSession,
     input_path="/Volumes/workspace/default/vol1/",  # the FOLDER, not the CSV file
    output_path="/Volumes/workspace/default/vol1/bronze/bronze_sales_transactions",
    checkpoint_path="/Volumes/workspace/default/vol1/_checkpoints/bronze",
    schema_location="/Volumes/workspace/default/vol1/_schema/bronze",
) -> DataFrame:
    """
    Production Bronze ingestion using Auto Loader (cloudFiles).

    - Incremental, exactly-once file discovery via cloudFiles
    - Schema inferred once and evolved (new columns), not
      re-inferred on every run
    - Bad records quarantined rather than failing the pipeline
    - Idempotent: checkpoint tracks processed files, safe to re-run
    """
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("badRecordsPath", f"{output_path}/_bad_records")
        .load(input_path)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", input_file_name())
    )

    query = (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .start(output_path)
    )
    query.awaitTermination()

    logger.info("Bronze ingestion complete: %s", output_path)
    return spark.read.format("delta").load(output_path)


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/vol1/"))

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("workspace.bronze.bronze_sales_transactions")
# Basic sanity checks before trusting the data downstream
row_count = bronze_df.count()
print(f"Row count: {row_count}")

bronze_df.printSchema()

display(
    bronze_df
    .orderBy(F.col("ingestion_timestamp").desc())
    .limit(20)
)
